In [ ]:
import numpy as np

# =====================================================
# CONSTANTS (Natural Units: GeV)
# =====================================================

ALPHA_EM = 1/137.036
MP = 0.938           # Proton mass
M_E = 0.000511       # Electron mass
PI = np.pi

# =====================================================
# VECTOR MESON PARAMETERS (light-quark states)
# =====================================================

VECTOR_MESONS = {
    "rho":   {"mass": 0.775, "fV": 5.0,  "width": 0.149},
    "omega": {"mass": 0.783, "fV": 17.0, "width": 0.00849}
}

# =====================================================
# SCALAR SPLITTING KERNEL
# =====================================================

def scalar_splitting_kernel(z):
    """
    Splitting function for vector -> scalar + vector
    """
    return z * (1 - z)


def scalar_radiation_probability(z, pT, alpha_dark):
    """
    Differential scalar radiation probability
    """
    kernel = scalar_splitting_kernel(z)
    return (alpha_dark / (2 * PI)) * kernel / (pT**2)


# =====================================================
# VMD PHOTOPRODUCTION
# gamma p -> V p -> S p
# =====================================================

def vmd_photoproduction_scalar(energy, m_s, g_sqq):
    """
    Scalar photoproduction via vector meson dominance.
    """

    s = 2 * MP * energy + MP**2
    sigma_total = 0

    for V in VECTOR_MESONS.values():

        mV = V["mass"]
        fV = V["fV"]
        width = V["width"]

        # photon-vector mixing
        photon_mixing = 4 * PI * ALPHA_EM / (fV**2)

        # scalar resonance structure
        propagator = 1 / ((m_s**2 - mV**2)**2 + mV**2 * width**2)

        # diffractive energy scaling
        sigma_V = photon_mixing * g_sqq**2 * propagator * s**0.08

        sigma_total += sigma_V

    return sigma_total


# =====================================================
# SCALAR RADIATION CORRECTION
# =====================================================

def corrected_photoproduction(energy, m_s, g_sqq, alpha_dark):

    sigma_base = vmd_photoproduction_scalar(energy, m_s, g_sqq)

    z_vals = np.linspace(0.01, 0.99, 100)

    P_emit = 0
    for z in z_vals:
        P_emit += scalar_radiation_probability(z, pT=0.1, alpha_dark=alpha_dark)

    P_emit *= (z_vals[1] - z_vals[0])

    sigma_total = sigma_base * (1 + P_emit)

    return sigma_total


# =====================================================
# EQUIVALENT PHOTON APPROXIMATION (EPA)
# =====================================================

def electron_photon_flux(z, Q2_max):

    term1 = (1 + (1 - z)**2) / z
    log_val = np.log(Q2_max * (1 - z) / (M_E**2 * z**2))

    return (ALPHA_EM / (2 * PI)) * term1 * log_val


# =====================================================
# TOTAL ELECTROPRODUCTION
# e p -> e p S
# =====================================================

def total_electroproduction(e_beam, m_s, g_sqq, alpha_dark):

    z_min = m_s**2 / (2 * MP * e_beam)
    z_vals = np.linspace(z_min, 0.99, 200)

    total_sigma = 0

    for z in z_vals:

        photon_energy = z * e_beam

        sigma_gamma = corrected_photoproduction(
            photon_energy,
            m_s,
            g_sqq,
            alpha_dark
        )

        flux = electron_photon_flux(z, Q2_max=1.0)

        total_sigma += sigma_gamma * flux

    total_sigma *= (z_vals[1] - z_vals[0])

    return total_sigma


# =====================================================
# EXAMPLE PARAMETERS
# =====================================================

m_scalar = 0.5     # GeV
g_light_quark = 1e-3
alpha_dark = 0.01
beam_energy = 12.0  # GeV (Jefferson Lab)

# =====================================================
# CALCULATIONS
# =====================================================

photo_xs = corrected_photoproduction(
    beam_energy,
    m_scalar,
    g_light_quark,
    alpha_dark
)

electro_xs = total_electroproduction(
    beam_energy,
    m_scalar,
    g_light_quark,
    alpha_dark
)

# =====================================================
# OUTPUT
# =====================================================

print("=== Scalar Photoproduction ===")
print(f"Photon-Proton Cross Section: {photo_xs:.4e} GeV^-2")

print("\n=== Scalar Electroproduction ===")
print(f"Electron-Proton Cross Section: {electro_xs:.4e} GeV^-2")